In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
from src import config as C

# 04 - Multilingual transformers (mBERT, XLM-RoBERTa)
Both models were fine-tuned and evaluated on a Google Colab **NVIDIA Tesla T4** (max 4 epochs, early stopping on validation macro-F1, patience 2, batch size 32). Training code: `src/train_transformer.py`; Colab procedure: `docs/COLAB_GPU.md`. This notebook displays the saved final results; it does not retrain. Values come from `results/verified_transformer_results.json` (owner-verified Colab summary) unless the raw run files are present in `experiments/results/`.

In [2]:
import json
sup = json.loads((C.ROOT_DIR / 'results' / 'verified_transformer_results.json').read_text())
print(sup['provenance'])
display(pd.DataFrame(sup['transformers']).T[['name','gpu','best_epoch','train_seconds','parameters','model_size_mb','gpu_latency_ms_per_message','cpu_latency_ms_per_message','accuracy','precision','recall','macro_f1','weighted_f1','macro_f1_ci95','robustness_degradation']])

Owner-supplied verified summary of the Google Colab (NVIDIA Tesla T4) transformer run and of the Colab QA run. The raw run files (experiments/results/transformer_<tag>.json, transformer_predictions_<tag>.csv, transformer_robustness_<tag>.csv, models/transformers/<tag>/) were NOT present in the sandbox where the final cleanup was performed, so these values could not be recomputed there. If those raw files are copied into experiments/results/, the code prefers them over this file and `python -m src.qa_check` recomputes and checks every value.


,name,gpu,best_epoch,train_seconds,parameters,model_size_mb,gpu_latency_ms_per_message,cpu_latency_ms_per_message,accuracy,precision,recall,macro_f1,weighted_f1,macro_f1_ci95,robustness_degradation
mbert,mBERT (bert-base-multilingual-cased),NVIDIA Tesla T4,2,287.5,177854978,681.3,9.52,125.65,0.9369,0.875,0.7226,0.8772,0.9344,"[0.754, 0.97]",-0.0055
xlmr,XLM-RoBERTa (xlm-roberta-base),NVIDIA Tesla T4,4,295.6,278045186,1077.0,10.25,117.48,0.985,0.9862,0.9226,0.9722,0.9848,"[0.946, 0.987]",0.0339


In [3]:
display(pd.DataFrame(sup['statistical_comparisons'])[['comparison','difference','ci95_low','ci95_high','ci_includes_zero','interpretation']])

,comparison,difference,ci95_low,ci95_high,ci_includes_zero,interpretation
0,"XLM-R vs selected Char TF-IDF + Linear SVM (overall test, macro-F1)",-0.001915,-0.018834,0.013255,True,No evidence of a difference; do not claim XLM-R superiority over the selecte...
1,"XLM-R vs mBERT (overall test, macro-F1)",0.095037,0.001722,0.223619,False,Interval excludes zero (lower bound barely above zero); XLM-R > mBERT on thi...
2,"XLM-R vs mBERT (English-only test, macro-F1)",0.008583,-0.006933,0.025362,True,No evidence of an English-only difference.


Smoke-test outputs (files/folders containing `_smoke`) are pipeline checks only and are never used for results.

In [4]:
from src import train_transformer as T
print('model zoo:', T.MODEL_ZOO)
print('protocol  :', sup['training_protocol'])

model zoo: {'mbert': 'bert-base-multilingual-cased', 'xlmr': 'xlm-roberta-base', 'distilmbert': 'distilbert-base-multilingual-cased'}
protocol  : {'max_epochs': 4, 'early_stopping': 'validation macro-F1', 'patience': 2, 'batch_size': 32, 'oom_fallback': 'batch size 16 (must be recorded explicitly)', 'defaults_of_the_documented_command': {'learning_rate': 2e-05, 'max_length': 128, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'seed': 42, 'loss': 'class-weighted cross-entropy', 'optimizer': 'AdamW, linear warm-up/decay'}}
